# graphArt_LM: scoring the hierarchical model (dev + generator test)

Runs the trained hierarchical adapter through `pipeline/run_hier.py` -- per program one tree
call, then one wiring call per block variant **of the model's own tree**, assembled by code --
and scores it with `pipeline/score_hier.py`: tree, variants, wiring, and the headline, the
flat graph after expansion against torch.fx. Three runs, two GPUs side by side:
- GPU 0: the 51 torchvision **dev** programs, then wiring alone on dev (`--oracle-tree`);
- GPU 1: the 75 held-out **generator v3** programs.
The frozen test sets are not in this notebook. About 1.5 h, capped at 2.5.

**Before running**: *Add Input* -> your dataset holding the adapter folder
`qwen2.5-coder-1.5b-graphart-lora-hier` (from `graphart_hier_train.zip`). *GPU T4 x2*,
*Internet on*. Then *Save Version -> Save & Run All (Commit)*.

**When it finishes**, download `graphart_hier_eval.zip` from the *Output* tab: the results
(`results/*.jsonl`, `.meta.json`, `.score.json`), the raw replies, and the logs.

Generated by `scripts/make_kaggle_notebook.py --hier-eval`. Edit that script, not this file.

In [ ]:
# BASE_MODEL   the base the adapter was trained on (its adapter_config.json names it too).
# ADAPTER      the adapter folder to look for under /kaggle/input (from graphart_hier_train.zip:
#              lora/qwen2.5-coder-1.5b-graphart-lora-hier/, uploaded as a dataset). If no folder
#              of that name is found, any single adapter under /kaggle/input is used.
# JOBS         one list per GPU, run in order on that GPU; the GPUs run at the same time.
#              (split, extra flags): hier_dev = the 51 torchvision dev programs, hier_gen_test =
#              75 held-out generator v3 programs, --oracle-tree = wiring alone (truth's tree).
# RUN_ARGS     output caps: the longest true tree is ~6,600 tokens, a wiring graph a few hundred.
# TIME_BUDGET_HOURS  the whole session; jobs still running at the end are stopped, and every
#              program finished so far is kept and scored.
# VALIDATE_ONLY True: only check the code arrived intact (works on a CPU session).
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"; ADAPTER = "qwen2.5-coder-1.5b-graphart-lora-hier"; GPUS = 2; JOBS = [[("hier_dev", []), ("hier_dev", ["--oracle-tree"])], [("hier_gen_test", [])]]; RUN_ARGS = ["--tree-tokens", "8192", "--wire-tokens", "2048"]; TIME_BUDGET_HOURS = 2.5; PACKAGE_RESERVE_MIN = 6; VALIDATE_ONLY = False

## 1. Unpack the code, prompts and samples

In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

START = time.time()
WORK = Path(os.environ.get("GRAPHART_WORK", "/kaggle/working"))
REPO = WORK / "graphArt_LM"
BUNDLE_SHA256 = "e1a85caefe7e583aa6d81a2db5868edf6f2ba5dd1e54d80dad165dba55978d5c"
BUNDLE = (
    "..."
    "FBc7MAMAAAAABFla"
)
data = base64.b64decode(BUNDLE)
assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256, "bundle damaged -- regenerate the notebook"
REPO.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(data), mode="r:xz") as tar:
    tar.extractall(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(sorted(str(p.relative_to(REPO)) for p in REPO.rglob("*") if p.is_file()))
if VALIDATE_ONLY:
    raise SystemExit("bundle intact -- VALIDATE_ONLY is set, so stopping here")
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

## 2. Install the pinned libraries

In [ ]:
from importlib.metadata import PackageNotFoundError, version

PINS = {'transformers': '5.20.2', 'peft': '0.21.2', 'accelerate': '1.15.0'}

def pinned(package, wanted):
    try:
        return version(package) == wanted
    except PackageNotFoundError:
        return False

missing = [f"{p}=={v}" for p, v in PINS.items() if not pinned(p, v)]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# peft probes two optional quantisation libraries when it builds LoRA layers, and
# raises -- rather than skipping them -- if they are installed but older than it
# supports. Kaggle ships torchao 0.10 (peft wants >= 0.16), which killed a run. We
# use neither, so remove them if present.
for pkg in ("torchao", "gptqmodel"):
    try:
        print(f"removing {pkg} {version(pkg)} (peft rejects old versions; not used here)", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg], check=True)
    except PackageNotFoundError:
        pass

subprocess.run([sys.executable, "-c", "import torch, transformers, peft; "
                "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
                "'| transformers', transformers.__version__, '| peft', peft.__version__)"])

## 3. Download the base model

In [ ]:
# Its own step with a time limit: a stalled download inside training would otherwise
# sit until the session cap. ~3 GB; a few minutes on Kaggle's network.
FETCH = ("from huggingface_hub import snapshot_download; "
         f"print('downloaded to', snapshot_download({BASE_MODEL!r}))")
for attempt in (1, 2):
    try:
        subprocess.run([sys.executable, "-c", FETCH], check=True, timeout=1800)
        break
    except (subprocess.TimeoutExpired, subprocess.CalledProcessError) as exc:
        print(f"download attempt {attempt} failed: {exc}", flush=True)
else:
    raise SystemExit("could not download the base model -- stopping before any training")

## 4. Find the adapter and the GPUs

In [ ]:
import zipfile

INPUT = Path(os.environ.get("GRAPHART_INPUT", "/kaggle/input"))
found = [c.parent for c in INPUT.rglob("adapter_config.json")] if INPUT.exists() else []
named = [p for p in found if p.name == ADAPTER]
if len(named) >= 1:
    adapter_src = named[0]
elif len(found) == 1:
    adapter_src = found[0]
else:
    raise SystemExit(f"no adapter folder {ADAPTER!r} under {INPUT} (found: {[str(p) for p in found]}) -- "
                     "add the dataset with the adapter as an input (see the top)")
# copied under its own name, so the results are named after it
adapter = REPO / "lora" / ADAPTER
if not adapter.exists():
    shutil.copytree(adapter_src, adapter)
print("adapter:", adapter_src, "->", adapter, flush=True)
summary = adapter / "train_summary.json"
if summary.exists():
    s = json.loads(summary.read_text())
    print(f"trained {s.get('steps')} steps, best val loss {s.get('best_eval_loss')}", flush=True)

n = int(subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.device_count())"],
                       capture_output=True, text=True).stdout.strip() or 0)
print(f"{n} GPU(s) visible", flush=True)
if n < 1:
    raise SystemExit("no GPU -- set the accelerator to GPU T4 x2")
if n < GPUS:
    print(f"only {n} GPU(s): running every job on GPU 0, one after another", flush=True)
    JOBS = [[job for jobs in JOBS for job in jobs]]
LOGS = REPO / "logs"
LOGS.mkdir(exist_ok=True)
ZIP_NAME = "graphart_hier_eval.zip"

def remaining():
    return TIME_BUDGET_HOURS * 3600 - (time.time() - START)

def package():
    with zipfile.ZipFile(WORK / ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as z:
        for root in (REPO / "results", LOGS):
            for p in root.rglob("*") if root.exists() else []:
                if p.is_file():
                    z.write(p, p.relative_to(REPO))

## 5. Run: one job list per GPU, side by side

In [ ]:
# Each GPU runs its list in one shell process; the lists run at the same time.
ENV = dict(os.environ, HF_HUB_OFFLINE="1", PYTHONUNBUFFERED="1")
procs = []
for gpu, jobs in enumerate(JOBS):
    script = []
    for split, extra in jobs:
        name = split + "".join(x.replace("--", "_") for x in extra)
        cmd = [sys.executable, "-m", "pipeline.run_hier", "--backend", "hf", "--model", str(adapter),
               "--split", split, *RUN_ARGS, *extra]
        script.append((name, cmd))
    log = open(LOGS / f"gpu{gpu}.log", "w")
    runner = ("import subprocess, sys\n"
              f"for name, cmd in {script!r}:\n"
              "    print(f'##### {name}', flush=True)\n"
              "    subprocess.run(cmd)\n")
    print(f"GPU {gpu}: {[n for n, _ in script]}", flush=True)
    procs.append((gpu, log, subprocess.Popen([sys.executable, "-c", runner], stdout=log, stderr=subprocess.STDOUT,
                                             env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu)))))
last = time.time()
while any(p.poll() is None for _, _, p in procs):
    if remaining() < PACKAGE_RESERVE_MIN * 60:
        for gpu, _, p in procs:
            if p.poll() is None:
                p.kill()
                print(f"GPU {gpu}: stopped by the time budget -- finished programs are kept", flush=True)
        break
    time.sleep(30)
    if time.time() - last > 600:            # a progress line every 10 minutes
        last = time.time()
        for gpu, _, _ in procs:
            lines = (LOGS / f"gpu{gpu}.log").read_text(errors="replace").splitlines()
            done = [l for l in lines if l.strip().startswith("[")]
            print(f"  GPU {gpu}: {done[-1].strip() if done else 'starting'}   ({remaining() / 3600:.1f} h left)",
                  flush=True)
for gpu, log, p in procs:
    log.close()
    print(f"\n##### GPU {gpu} (exit {p.returncode})\n" +
          "\n".join((LOGS / f"gpu{gpu}.log").read_text(errors="replace").splitlines()[-25:]), flush=True)
package()

## 6. Score

In [ ]:
runs = sorted(str(p.relative_to(REPO)) for p in (REPO / "results").glob("hf__*__hier*.jsonl"))
out = subprocess.run([sys.executable, "-m", "pipeline.score_hier", *runs], capture_output=True, text=True)
print(out.stdout[-20000:], out.stderr[-3000:], flush=True)
(LOGS / "scores.txt").write_text(out.stdout)
package()
print(f"done in {(time.time() - START) / 3600:.1f} h -- download {ZIP_NAME}")